# 🎓 WE4 · Notebook 04: PPO
## Teaching an agent to play Flappy Bird

This notebook trains an agent with PPO, the algorithm from the lecture. You fill
in four short blanks: the critic network, and the three lines of PPO's objective.
Everything else is already here.

By the end you will have a video of an agent that could not survive one second
learning to fly through pipes.

### Before you start

**Runtime, then Change runtime type, then choose CPU.** Not GPU.

The network here is tiny (12 inputs, two hidden layers of 64) and the agent plays
one step at a time, so a GPU only adds launch latency to every one of a hundred
thousand tiny calls. It would be slower, and it would spend your GPU quota.

### The plan

| | |
|---|---|
| 1 | Set up. Four cells to run, nothing to read |
| 2 | Meet the environment: how you talk to a game |
| 3 | Write the critic network, then watch an untrained agent fail |
| 4 | Write the three lines of PPO's objective, and check them in one second |
| 5 | Train for about two minutes |
| 6 | Watch the result |

## 1. Setup

The next four cells are plumbing: installing, importing, and defining helpers to
turn frames into video. **Run them and move on.** Nothing in them is part of the
exercise, and none of it is examinable. Click "Show code" if you are curious.

In [ ]:
#@title Run me: install
# This installs almost nothing: Colab already ships gymnasium, pygame and torch.
# The pins are here so the notebook still works if Google changes the image.
# Do not add -q: it would hide a "RESTART SESSION" banner if that ever happens.
!pip install "gymnasium==1.3.0" "flappy-bird-gymnasium==0.4.0"

In [ ]:
#@title Run me: version check
# Version guard. If anything goes wrong later, the output of this cell is the
# first thing to look at.
import gymnasium, numpy, torch, flappy_bird_gymnasium
print("gymnasium", gymnasium.__version__)
print("numpy    ", numpy.__version__)
print("torch    ", torch.__version__)
assert gymnasium.__version__.startswith("1."), "expected gymnasium 1.x"
print("\nOK")

In [ ]:
#@title Run me: imports, and one frame of the game
import numpy as np, torch, torch.nn as nn, time
import gymnasium as gym
import flappy_bird_gymnasium   # registers FlappyBird-v0

torch.set_num_threads(1)       # measured: no slower, and it keeps runs reproducible

def make_env(render=False):
    return gym.make("FlappyBird-v0",
                    render_mode="rgb_array" if render else None,
                    use_lidar=False)

env = make_env(render=True)
print("state :", env.observation_space)
print("action:", env.action_space)

obs, _ = env.reset(seed=0)
frame = env.render()
print("one rendered frame:", frame.shape)
env.close()

import matplotlib.pyplot as plt
plt.figure(figsize=(3, 5)); plt.imshow(frame); plt.axis("off"); plt.show()

In [ ]:
#@title Run me: a helper that turns frames into video
from base64 import b64encode
from IPython.display import HTML
import imageio

def save_video(frames, path, fps=30):
    '''Write frames to mp4, falling back to gif if ffmpeg is unavailable.'''
    try:
        imageio.mimsave(path, frames, fps=fps, macro_block_size=1)
        return path
    except Exception as e:
        print("mp4 failed (%s), falling back to gif" % type(e).__name__)
        gif = path.replace(".mp4", ".gif")
        imageio.mimsave(gif, frames[::2], duration=1000 / (fps / 2), loop=0)
        return gif

def show_video(path, width=260, caption=""):
    data = b64encode(open(path, "rb").read()).decode()
    if path.endswith(".gif"):
        tag = '<img width="%d" src="data:image/gif;base64,%s">' % (width, data)
    else:
        tag = ('<video width="%d" autoplay loop controls>'
               '<source src="data:video/mp4;base64,%s" type="video/mp4"></video>') % (width, data)
    return '<figure style="margin:0 12px 0 0;text-align:center">%s<figcaption style="font:13px sans-serif;color:#555">%s</figcaption></figure>' % (tag, caption)

def play(*figs):
    display(HTML('<div style="display:flex;align-items:flex-start">%s</div>' % "".join(figs)))

## 2. The environment

Everything the lecture defined has a concrete meaning here.

| Lecture | Flappy Bird |
|---|---|
| **state** | 12 numbers: the bird's height and speed, and the positions of the next pipes |
| **action** | 2 choices: `0` do nothing, `1` flap |
| **reward** | `+0.1` for staying alive one frame, `+1.0` for passing a pipe, `-1.0` for dying |
| **episode** | one life, from the start until the bird hits a pipe or the ground |
| **return** | everything collected in one life |

### How you talk to it

Two methods do everything.

`env.reset()` starts a new life and hands back the first state.

`env.step(action)` takes one action and hands back **five** things:

| what comes back | what it means | in Flappy Bird |
|---|---|---|
| `observation` | the new state, after your action | the 12 numbers again, updated |
| `reward` | what that single action earned | `+0.1`, or `+1.0` at a pipe, or `-1.0` if it just died |
| `terminated` | did the episode end **because of the task** | the bird crashed |
| `truncated` | was the episode cut short **from outside**, e.g. a time limit | never happens in this game |
| `info` | extra diagnostics, ignored by the algorithm | the score so far |

`terminated` and `truncated` are separate because they mean different things to
the algorithm. When an episode *terminates* there is genuinely no future left to
value. When it is merely *truncated* the future still exists, it just stopped
being recorded. This game only ever terminates, so the code below treats them
together.

### The methods you will see

| method | what it does |
|---|---|
| `env.reset(seed=...)` | start a new life, and hand back `(observation, info)` |
| `env.step(action)` | take one action, and hand back the five things above |
| `env.render()` | hand back the current frame as pixels, shape `(512, 288, 3)` |
| `env.action_space.sample()` | a random legal action, useful before there is a policy |
| `env.close()` | let the game go |

The whole of reinforcement learning is a loop over `reset` and `step`.

In [ ]:
env = make_env()

obs, info = env.reset(seed=0)
print("env.reset() gives the first state:")
print("   observation:", np.round(obs, 3))
print("   info       :", info)

obs, reward, terminated, truncated, info = env.step(1)      # 1 = flap
print("\nenv.step(1) gives five things back:")
print("   observation:", np.round(obs, 3))
print("   reward     :", reward)
print("   terminated :", terminated)
print("   truncated  :", truncated)
print("   info       :", info)

### One whole life, played at random

The loop below is the shape of every reinforcement learning program ever written:
act, observe, add up the reward, stop when the episode ends.

In [ ]:
obs, _ = env.reset(seed=0)
total_reward, steps = 0.0, 0

while True:
    action = env.action_space.sample()            # a coin flip, no policy yet
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    steps += 1
    if terminated or truncated:
        break

print("a random agent survived %d steps (%.1f seconds)" % (steps, steps / 30))
print("its RETURN for that life was %.2f" % total_reward)
print("\nthat return is the number the whole algorithm is trying to make bigger.")
env.close()

## 3. An untrained agent

One line in the next cell is worth reading slowly, because it is how a network
turns into a decision:

```python
a = int(torch.argmax(net.actor(torch.as_tensor(obs, dtype=torch.float32))))
```

Inside out:

1. `torch.as_tensor(obs, dtype=torch.float32)` turns the 12 numbers into a tensor
   the network can accept. The cast matters: the weights are float32 and torch
   will not quietly mix types.
2. `net.actor(...)` pushes them through the network. Out come **two numbers, one
   per action**. They are *logits*, unbounded scores, not probabilities. Higher
   means the policy likes that action more.
3. `torch.argmax(...)` takes the index of the larger one, so `0` or `1`.
4. `int(...)` unwraps it to a plain integer, which is what `env.step` wants.

It is wrapped in `with torch.no_grad():`, which tells torch **not to record any
of this for training**. While playing we only want the answer, not the machinery
for differentiating it. That is also why the log-probabilities stored during
play are frozen constants later: they were computed with the gradient turned off.

### Your first blank: the critic

The lecture gave the two jobs. Here they are as two small networks, and the
difference between them is a single number.

The **actor** answers *"how much do I like each action here?"*, so a state goes in
and **one number per action** comes out.

The **critic** answers *"how good is this state?"*, so a state goes in and
**exactly one number** comes out. One. Not one per action, because the critic
does not score actions at all: it scores the situation.

The actor is written for you below. Write the critic. It is the same shape,
right up to the last layer.

In [ ]:
class ActorCritic(nn.Module):
    '''Two small networks. The actor picks actions; the critic scores states.'''
    def __init__(self, n_obs=12, n_act=2):
        super().__init__()

        # THE ACTOR: n_obs numbers in, two hidden layers of 64 with tanh,
        # and n_act numbers out, one score per action.
        self.actor  = nn.Sequential(nn.Linear(n_obs, 64), nn.Tanh(),
                                    nn.Linear(64, 64),    nn.Tanh(),
                                    nn.Linear(64, n_act))

        # THE CRITIC: identical, except for how many numbers come out.
        # Re-read the paragraph above if you are unsure what that number is.
        self.critic = None                        # <<<<<< replace this

    def dist(self, obs):   return torch.distributions.Categorical(logits=self.actor(obs))
    def value(self, obs):  return self.critic(obs).squeeze(-1)


def rollout(net, seed=0, max_frames=1200):
    '''Play one life with the agent's best-guess action and record the frames.'''
    env = make_env(render=True)
    obs, _ = env.reset(seed=seed)
    frames, total, pipes = [], 0.0, 0
    for _ in range(max_frames):
        frames.append(env.render())
        with torch.no_grad():
            a = int(torch.argmax(net.actor(torch.as_tensor(obs, dtype=torch.float32))))
        obs, r, term, trunc, _ = env.step(a)
        total += float(r)
        if r >= 1.0: pipes += 1
        if term or trunc: break
    env.close()
    return frames, total, pipes

In [ ]:
#@title Run me: check your critic
def check_critic():
    try:
        net = ActorCritic()
    except Exception as e:
        print("Building the network raised %s: %s" % (type(e).__name__, e)); return

    if net.critic is None:
        print("FAIL: self.critic is still None. Write it, copying the actor above.")
        return

    try:
        states = torch.zeros(5, 12)          # five pretend states
        raw = net.critic(states)
        v = net.value(states)
    except Exception as e:
        print("Your critic raised %s: %s" % (type(e).__name__, e))
        print("Check that the first layer accepts n_obs inputs.")
        return

    if tuple(raw.shape) != (5, 1):
        print("FAIL: given 5 states, your critic returned shape %s." % (tuple(raw.shape),))
        print("      It should return (5, 1): exactly one number per state.")
        if raw.shape[-1] == 2:
            print("      You gave it n_act outputs. The critic does not score actions,")
            print("      it scores the STATE, with one number however many actions exist.")
        return

    print("PASS. Your critic turns a state into a single number.")
    print("      actor  output for one state:", tuple(net.actor(torch.zeros(12)).shape), "(one score per action)")
    print("      critic output for one state:", tuple(net.critic(torch.zeros(12)).shape), "(one value, full stop)")

check_critic()

In [ ]:
SEED = 0
torch.manual_seed(SEED); np.random.seed(SEED)

untrained = ActorCritic()
frames_before, ret_before, pipes_before = rollout(untrained, seed=SEED)
print("untrained: survived %d frames (%.1f seconds), %d pipes, return %.2f"
      % (len(frames_before), len(frames_before) / 30, pipes_before, ret_before))

before_path = save_video(frames_before, "before.mp4")
play(show_video(before_path, caption="before training"))

## 4. Your task: PPO's objective

Everything so far was setup. This is the exercise.

### Step 1: the probability ratio

The 2048 steps were played by the policy **as it was before this update**. As soon
as you improve the policy once, that data was collected by somebody who no longer
exists. The probability ratio is how you keep using it anyway:

$$\rho_t(\theta) \;=\; \frac{\pi_\theta(a_t \mid s_t)}{\pi_{\theta_{\mathrm{old}}}(a_t \mid s_t)}$$

It asks: **how much more, or less, often would the policy I am building now have
taken this action?**

- $\rho_t > 1$: the new policy favours this action more than the collector did.
  The sample is more representative of the new policy, so it counts for more.
- $\rho_t < 1$: the new policy has moved away from this action. The sample says
  less about the new policy, so it counts for less.
- $\rho_t = 1$: the two policies agree here, and nothing is reweighted.

You are given the logarithms, `new_logp` and `old_logp`. A ratio is the
exponential of the difference of logarithms.

### Step 2: the clipped ratio

Left alone, that ratio is unbounded, and one sample could drag the policy
anywhere. So cap it:

$$\bar{\rho}_t \;=\; \operatorname{clip}(\rho_t,\; 1-\epsilon,\; 1+\epsilon)$$

If it is too large, it becomes $1+\epsilon$. If it is too small, it becomes
$1-\epsilon$. In torch this is `torch.clamp(x, lo, hi)`.

### Step 3: keep the pessimistic one

Now there are two possible multipliers for the same sample: the honest one
$\rho_t \widehat{A}_t$, and the capped one $\bar{\rho}_t \widehat{A}_t$.
PPO **takes the smaller of the two**:

$$L \;=\; \operatorname{average}\Big[\min\big(\rho_t \widehat{A}_t,\;\; \bar{\rho}_t \widehat{A}_t\big)\Big]$$

Taking the **minimum** means always believing the less flattering of the two
estimates. That one word is what makes the clipping do the right thing:

| advantage positive, and the ratio has | the clipped term is | `min` keeps | effect |
|---|---|---|---|
| grown past $1+\epsilon$ | smaller | the cap | no further reward for pushing |
| fallen below $1-\epsilon$ | larger | the honest $\rho_t \widehat{A}_t$ | the gradient still flows, so it can come back |

So clipping stops a sample pushing **further away** from $\rho = 1$, but never
stops it coming **back**. With `max` instead of `min` you would get exactly the
wrong behaviour in both rows.

One last thing: we want to **maximise** $L$, but optimisers **minimise**, so
return the negative.

Fill in the three lines below.

In [ ]:
CLIP_EPS = 0.2

def ppo_losses(new_logp, old_logp, adv, clip_eps=CLIP_EPS):
    '''
    new_logp : log of the probability the CURRENT policy gives the action taken
    old_logp : log of the probability the policy that COLLECTED the data gave it
    adv      : the advantage estimate for each step

    Returns (ratio, policy_loss).
    '''

    # STEP 1. The probability ratio.
    #         Hint: exp(log a - log b) = a / b
    ratio = None            # <<<<<< replace this

    # STEP 2. The same ratio, capped below at 1-clip_eps and above at 1+clip_eps.
    #         Hint: torch.clamp(x, lo, hi)
    clipped_ratio = None    # <<<<<< replace this

    # STEP 3. Two candidate multipliers, ratio*adv and clipped_ratio*adv.
    #         Keep the SMALLER of the two for each step, average over the batch,
    #         and negate so it can be minimised.
    #         Hint: -torch.min(A, B).mean()
    policy_loss = None      # <<<<<< replace this

    return ratio, policy_loss

### Check your three lines, before training anything

This runs in about a second on fixed numbers, and it names the specific
mistake. Do not start the two-minute training run until it says PASS.

In [ ]:
#@title Run me: check your three lines
def check():
    new_logp = torch.tensor([-0.30, -1.20, -0.05, -2.00, -0.70])
    old_logp = torch.tensor([-0.50, -0.90, -0.05, -1.10, -1.40])
    adv      = torch.tensor([ 1.50, -0.80,  2.00,  0.60, -1.30])

    try:
        ratio, loss = ppo_losses(new_logp, old_logp, adv, 0.2)
    except Exception as e:
        print("Your code raised %s: %s" % (type(e).__name__, e)); return

    if ratio is None or loss is None:
        print("FAIL: something is still None. All three steps need replacing."); return

    want_ratio = torch.tensor([1.221403, 0.740818, 1.0, 0.40657, 2.013753])
    if not torch.allclose(ratio, want_ratio, atol=1e-4):
        print("FAIL on STEP 1, the ratio.")
        print("  you     :", [round(float(x), 4) for x in ratio])
        print("  expected:", [round(float(x), 4) for x in want_ratio])
        print("  It must be 1.0 wherever new_logp equals old_logp. Check the order")
        print("  of the subtraction: it is new minus old.")
        return

    v = float(loss)
    if abs(v - (-0.157213)) < 1e-4:
        print("PASS. All three steps are right. Go and train.")
        print("(for reference, the clipped ratios here are [1.2, 0.8, 1.0, 0.8, 1.2])")
        return

    print("STEP 1 is right, but the final loss is not.")
    print("  you got %.6f, expected -0.157213" % v)
    if abs(v - 0.157213) < 1e-4:
        print("  Right number, wrong sign. Negate it.")
    elif abs(v - (-0.43189)) < 1e-4:
        print("  That is max() instead of min(). PPO keeps the SMALLER of the two")
        print("  candidates, the pessimistic one.")
    elif abs(v - (-0.173103)) < 1e-4:
        print("  That is ratio*adv only: STEP 2 is not being used in STEP 3.")
    elif abs(v - (-0.416)) < 1e-3:
        print("  That is clipped_ratio*adv only: you need BOTH candidates.")
    else:
        print("  Expected: -torch.min(ratio*adv, clipped_ratio*adv).mean()")

check()

## 5. Train

**You do not write anything here.** Read the shape, run the cell, watch the
number climb. The loop below is ordinary PPO, and it calls the `ppo_losses` you
just wrote.

In pseudocode, the whole cell is three phases repeated fifty times:

```
repeat 50 times:                      # one "update"

  1. PLAY
     play 2048 steps with the CURRENT policy, and for each step write down:
        state, action, log-probability of that action, the critic's value,
        the reward, and whether the life ended

  2. SCORE
     walk BACKWARDS through those 2048 steps:
        surprise  = reward + discount x value(next state) - value(this state)
        advantage = surprise + discount x lambda x advantage(next step)

  3. IMPROVE
     repeat 10 times:                 # ten passes over the SAME 2048 steps
        for each minibatch of 64:
           loss = YOUR THREE LINES
                + 0.5 x (critic's error)^2      # teach the critic to predict
                - 0.01 x (policy's entropy)     # pay it to stay undecided
           nudge both networks downhill

  throw the 2048 steps away: the policy has moved, so they are stale
```

That `repeat 10 times` is the reason PPO exists. Without the probability ratio
you would be allowed **one** pass over the data before it went stale, and those
2048 hard-won steps would buy a single update. Your three lines are what buys the
other nine. Over the whole run that is 16,000 gradient steps out of 102,400 steps
of the game, instead of 50.

The loop below is ordinary PPO: play 2048 steps, work out how good each action
was, then improve the policy with your three lines. It repeats that 50 times, which
is 102,400 steps of the game.

Everything here was given in the lecture except the three lines you just wrote.

**About two minutes** (measured: 117 seconds on a Colab CPU runtime). Watch the
mean return climb. It is bumpy, and that is the point of the next section.

In [ ]:
UPDATES, ROLLOUT, EPOCHS, MINIBATCH = 50, 2048, 10, 64
GAMMA, LAM, LR, ENT_COEF, VF_COEF = 0.99, 0.95, 3e-4, 0.01, 0.5

def train(seed=SEED, updates=UPDATES):
    torch.manual_seed(seed); np.random.seed(seed)     # re-seed here so re-running is repeatable
    env = make_env()
    net = ActorCritic(env.observation_space.shape[0], env.action_space.n)
    opt = torch.optim.Adam(net.parameters(), lr=LR)

    obs, _ = env.reset(seed=seed)
    obs = torch.as_tensor(obs, dtype=torch.float32)
    ep_ret, log, curve = 0.0, [], []
    t0 = time.time()

    for update in range(updates):
        O = torch.zeros(ROLLOUT, obs.shape[0]); A = torch.zeros(ROLLOUT, dtype=torch.long)
        LOGP = torch.zeros(ROLLOUT); R = torch.zeros(ROLLOUT)
        D = torch.zeros(ROLLOUT);    V = torch.zeros(ROLLOUT)

        for t in range(ROLLOUT):                      # play
            with torch.no_grad():
                d = net.dist(obs); a = d.sample()
                O[t], A[t], LOGP[t], V[t] = obs, a, d.log_prob(a), net.value(obs)
            nobs, r, term, trunc, _ = env.step(int(a))
            R[t], D[t] = float(r), float(term or trunc)
            ep_ret += float(r)
            if term or trunc:
                log.append(ep_ret); ep_ret = 0.0; nobs, _ = env.reset()
            obs = torch.as_tensor(nobs, dtype=torch.float32)

        with torch.no_grad(): last_v = net.value(obs)  # how good was each action
        adv, gae = torch.zeros(ROLLOUT), 0.0
        for t in reversed(range(ROLLOUT)):
            nextv = last_v if t == ROLLOUT - 1 else V[t + 1]
            nonterm = 1.0 - D[t]
            delta = R[t] + GAMMA * nextv * nonterm - V[t]
            gae = delta + GAMMA * LAM * nonterm * gae
            adv[t] = gae
        ret = adv + V

        idx = np.arange(ROLLOUT)                       # improve the policy
        for _ in range(EPOCHS):
            np.random.shuffle(idx)
            for s in range(0, ROLLOUT, MINIBATCH):
                mb = idx[s:s + MINIBATCH]
                d = net.dist(O[mb])
                a_mb = adv[mb]; a_mb = (a_mb - a_mb.mean()) / (a_mb.std() + 1e-8)

                _, policy_loss = ppo_losses(d.log_prob(A[mb]), LOGP[mb], a_mb)   # <<< your code

                value_loss = ((net.value(O[mb]) - ret[mb]) ** 2).mean()
                loss = policy_loss + VF_COEF * value_loss - ENT_COEF * d.entropy().mean()
                opt.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(net.parameters(), 0.5)
                opt.step()

        recent = float(np.mean(log[-20:])) if log else 0.0
        curve.append(recent)
        if (update + 1) % 5 == 0:
            print("update %2d/%d   steps %6d   mean return %6.2f   %4.0fs"
                  % (update + 1, updates, (update + 1) * ROLLOUT, recent, time.time() - t0), flush=True)
    env.close()
    return net, curve

trained, curve = train()

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(7, 3))
plt.plot(np.arange(1, len(curve) + 1) * ROLLOUT, curve)
plt.xlabel("steps of the game played"); plt.ylabel("mean return, last 20 lives")
plt.title("learning curve"); plt.grid(alpha=.3); plt.show()

## 6. Watch it play

The clip below is recorded with the agent's **best** action at every step rather
than a sampled one, so what you see is what it has actually learned and not a
lucky or unlucky draw.

In [ ]:
frames_after, ret_after, pipes_after = rollout(trained, seed=SEED)
print("trained  : survived %d frames (%.1f seconds), %d pipes, return %.2f"
      % (len(frames_after), len(frames_after) / 30, pipes_after, ret_after))
print("untrained: survived %d frames (%.1f seconds), %d pipes, return %.2f"
      % (len(frames_before), len(frames_before) / 30, pipes_before, ret_before))

after_path = save_video(frames_after, "after.mp4")
play(show_video(before_path, caption="before: %d pipes" % pipes_before),
     show_video(after_path,  caption="after: %d pipes"  % pipes_after))

## What to notice

**The reward never told it how to fly.** It only ever said `+0.1` for surviving a
frame and `+1.0` for a pipe. Nobody wrote a rule about when to flap. The flying
came out of trying and keeping what worked.

**The learning curve is bumpy.** That is normal and it is the subject of the
lecture: each update is measured from a noisy sample, and PPO's clipping is what
stops one bad sample throwing the policy away.

**It is not finished.** Three minutes of training on a laptop-sized machine buys
a competent beginner, not an expert. The published agents that play forever train
for tens of millions of steps.

### If you have time

- Set `CLIP_EPS = 1000.0` (effectively no clipping) and train again. This is the
  experiment the lecture describes: watch the run become unstable.
- Set `ENT_COEF = 0.0`. That removes the reward for staying undecided. The policy
  often collapses onto one action early and then stops improving.
- Change `SEED` and re-run. The agent is not identical every time.